# D1 PCA — privacy-utility trade-off via channel compression

Defender fits PCA on the channel dimension of the training EEG, projects every window to top-k components, then trains the BCI decoder on the projected data. As k drops, both task accuracy AND re-ID accuracy should drop. The shape of that joint trajectory is the defense's quality.

Sweeps k ∈ {64 (no defense), 32, 16, 8} on PhysioNet motor imagery, 104 subjects, all 3 victim families. EEGNet uses 40 epochs (vs 80 in A1) so the four-condition sweep fits the 1-hour budget.

Send back `results/07_d1_pca.json` and `runs/<run_id>/meta.json`.

**Runtime → L4 GPU.** Expected wall ~50 min.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2 min)

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run D1 PCA sweep — all 3 victims × 4 k values

Expected wall: ~50 min.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.07_d1_pca --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch

PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

EXPERIMENT = 'experiments.07_d1_pca'
ARGS = ['--all']
RESULTS = 'results/07_d1_pca.json'
FIGURE = 'figures/07_d1_pca.pdf'
TAG = 'd1_pca'

try:
    git_sha = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR
    ).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"')
    git_sha = 'unknown'

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'

meta = {
    'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
    'git_sha': git_sha, 'hardware': f'Colab {gpu}',
    'platform': platform.platform(), 'torch_version': torch.__version__,
    'completed_at_utc': now_utc, 'outputs': [RESULTS, FIGURE],
}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('=== run metadata ===')
print(json.dumps(meta, indent=2))
print()

if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — Cell 5 did not finish. Re-run Cell 5, then this cell.')
print(f'=== {RESULTS} ===')
print(open(RESULTS).read())

## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/07_d1_pca.json')
files.download(f'runs/{run_id}/meta.json')